# FlashNystrom — Colab experiments → paper artifacts (saved to Drive)

Runs the paper experiments and saves **everything to Google Drive**: the raw JSON (every plotted point), full console logs (`.log`, per-epoch / per-LR detail), the PDF+PNG figures, and a zip. Re-style plots anytime by re-running section 8 on the saved JSON.

**GPU:** kernels are **sm_80+** — use **A100 or L4** (Colab Pro/Pro+). A **T4 will NOT work.**

**Multi-session:** Colab wipes the VM between sessions, so re-run cells 0–4 (clone+build+Drive) each session; results persist in the fixed Drive folder, so sections run across sessions accumulate and `make_figures` picks them all up.

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name} sm_{cap[0]}{cap[1]}. Switch runtime.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
# Fresh, single clone (avoids nested-clone if this cell is re-run) WITH submodules
%cd /content
!rm -rf flashnystrom
!git clone --recurse-submodules $REPO_URL flashnystrom
%cd flashnystrom
!git submodule update --init --recursive   # ensure CUTLASS (CuTe) headers
import os
assert os.path.isdir('third_party/cutlass/include'), \
    'CUTLASS submodule did not fetch -- re-run this cell (check network).'
print('CUTLASS headers present - ok to build')

## 2. Build the fused CUDA kernels (~3–6 min)

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
# Colab's gcc/CUDA toolchain trips -Werror on third-party (CUTLASS/torch) headers
# that the author's build doesn't; LAX disables the warnings-as-errors guard.
os.environ['FLASH_NYSTROM_LAX_BUILD'] = '1'
!pip install -e . --no-build-isolation

## 3. Verify the kernels

In [ ]:
import torch
from flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference
mk = lambda: torch.randn(4, 2, 256, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
o = flash_nystrom_attention(q, k, v, 64, 6); r = nystrom_attention_reference(q, k, v, 64, 6)
print('fwd finite:', bool(torch.isfinite(o).all()), ' max|fn-ref|:', (o.float()-r.float()).abs().max().item())
qg = q.clone().requires_grad_(True); flash_nystrom_attention(qg, k, v, 64, 6).sum().backward()
print('bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Mount Drive + pick an output folder
Everything (JSON, logs, figures, zip) goes here. **Re-run this each session**; keep `RUN_NAME` the same to accumulate, or change it for a fresh run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_NAME = 'flashnystrom_run'   # change for a separate run
OUTDIR = '/content/drive/MyDrive/flashnystrom_runs/' + RUN_NAME
os.makedirs(OUTDIR + '/figures', exist_ok=True)
print('All results ->', OUTDIR)

## 5. Scaling / crossover  → `scaling.json` + `scaling.log`
Throughput + peak memory vs N at the auto-found max batch (saturates the GPU). (~10–20 min)

In [ ]:
cmd = ('python benchmarks/profile_scaling.py'
       ' --backends sdpa flash_nystrom nystrom_reference'
       ' --Ns 256 512 1024 2048 4096 8192 16384'
       f' --json {OUTDIR}/scaling.json 2>&1 | tee {OUTDIR}/scaling.log')
!{cmd}

## 6. MQAR recall  → `mqar_length.json`, `mqar_capacity.json` (+ logs)
Length sweep (recall vs context + faithfulness) and capacity ablation (rank limit). Validated fixed-batch recipe + grad_clip. Longest step — trim lists to iterate.

In [ ]:
cmd = ('python -m paper.mqar.run_scaling_sweep --mode length'
       ' --backends sdpa flash_nystrom nystrom_reference'
       ' --seq_lens 256 512 1024 2048 4096 --num_kv_pairs 16'
       f' --json {OUTDIR}/mqar_length.json 2>&1 | tee {OUTDIR}/mqar_length.log')
!{cmd}

In [ ]:
cmd = ('python -m paper.mqar.run_scaling_sweep --mode capacity'
       ' --backends sdpa flash_nystrom nystrom_reference'
       ' --seq_len 1024 --kv_pairs 16 32 64 128 256'
       f' --json {OUTDIR}/mqar_capacity.json 2>&1 | tee {OUTDIR}/mqar_capacity.log')
!{cmd}

## 7. CIFAR pixel-token  → `three_way_results.json` + log
`patch_size=1` → 1025-token sequence, fixed batch + grad_clip. (~30–60 min)

In [ ]:
cmd = ('python benchmarks/train_three_way.py'
       ' --patch_size 1 --epochs 30 --grad_clip 1.0'
       ' --backends sdpa nystrom_reference flash_nystrom'
       f' 2>&1 | tee {OUTDIR}/cifar.log')
!{cmd}
!cp three_way_results.json {OUTDIR}/three_way_results.json

## 8. Build figures (PDF + PNG) from the saved JSON
Re-run anytime to re-style — only reads the JSON, no GPU needed.

In [ ]:
cmd = ('python benchmarks/make_figures.py'
       f' --scaling {OUTDIR}/scaling.json'
       f' --mqar_length {OUTDIR}/mqar_length.json'
       f' --mqar_capacity {OUTDIR}/mqar_capacity.json'
       f' --cifar {OUTDIR}/three_way_results.json'
       f' --outdir {OUTDIR}/figures')
!{cmd}

## 9. Inventory + zip (already on Drive)

In [ ]:
import glob, zipfile
arts = sorted(glob.glob(OUTDIR+'/figures/*') + glob.glob(OUTDIR+'/*.json') + glob.glob(OUTDIR+'/*.log'))
print('Saved to Drive:', OUTDIR)
for p in arts: print('  ' + os.path.relpath(p, OUTDIR))
zf = OUTDIR + '/artifacts.zip'
with zipfile.ZipFile(zf, 'w') as z:
    for p in arts: z.write(p, os.path.relpath(p, OUTDIR))
print('zipped ->', zf)
# local copy too: from google.colab import files; files.download(zf)

## Optional — 3-seed error bars on MQAR recall (heads=2)
The figures use one seed (best-over-LR). For mean±std on the headline recall, run this (slower); output is logged to Drive.

In [ ]:
cmd = ('python -m paper.mqar.sweep'
       ' --backends sdpa flash_nystrom nystrom_reference'
       ' --heads 2 --inits normal --seeds 0 1 2'
       ' --lrs 1e-3 3.16e-3 1e-2 3.16e-2 --grad_clip 1.0 --epochs 64'
       f' 2>&1 | tee {OUTDIR}/recall_3seed.log')
!{cmd}

## Notes
- **Nothing is lost on disconnect** — JSON/logs/figures live on Drive in `OUTDIR`.
- Re-style plots with no GPU: re-run section 8 (reads the saved JSON only).
- Run one heavy section per session to economize CUs; JSON accumulates in `OUTDIR`.
- Preview a figure inline: `from IPython.display import Image; Image(OUTDIR+'/figures/scaling.png')`.